## **Aim**
To implement a program that generates an incident response timeline from collected digital evidence and investigation records.

## **Algorithm**
**Step 1:** Import `json`, `datetime`, `collections` libraries.

**Step 2:** Create simulated evidence sources:
   - File system timeline (creation, modification, access times)
   - Network logs (connections, DNS, HTTP)
   - Process execution logs
   - Authentication logs
   - Registry changes
   - Memory artifacts

**Step 3:** Normalize all events to common format: timestamp, source, category, description, severity, evidence_ref.

**Step 4:** Merge and sort all events chronologically.

**Step 5:** Correlate related events across sources using time proximity and shared indicators (IPs, users, files).

**Step 6:** Identify key phases: Initial Access, Execution, Persistence, Privilege Escalation, Defense Evasion, Credential Access, Discovery, Lateral Movement, Collection, Exfiltration, Impact.

**Step 7:** Generate structured timeline report with phase markers and correlations.

In [1]:
import json
from datetime import datetime, timedelta
from collections import defaultdict

def create_sample_evidence():
    now = datetime.now()
    base = now - timedelta(hours=7)
    
    events = [
        # Phase 1: Initial Access
        {"timestamp": (base + timedelta(minutes=0)).isoformat(), "source": "EMAIL", "category": "INITIAL_ACCESS",
         "description": "Phishing email delivered to user@company.com",
         "severity": "HIGH", "evidence": "email_gateway.log",
         "iocs": ["paypal-security-update.tk", "Invoice_2026-0820.docx"]},
        {"timestamp": (base + timedelta(minutes=5)).isoformat(), "source": "ENDPOINT", "category": "INITIAL_ACCESS",
         "description": "User opened malicious document, macro executed",
         "severity": "HIGH", "evidence": "edr.log",
         "iocs": ["WINWORD.EXE -> powershell.exe (PID: 4521)"]},
        
        # Phase 2: Execution
        {"timestamp": (base + timedelta(minutes=10)).isoformat(), "source": "NETWORK", "category": "EXECUTION",
         "description": "PowerShell downloaded Cobalt Strike beacon",
         "severity": "CRITICAL", "evidence": "proxy.log",
         "iocs": ["http://192.168.100.50/beacon.ps1"]},
        {"timestamp": (base + timedelta(minutes=15)).isoformat(), "source": "ENDPOINT", "category": "EXECUTION",
         "description": "Beacon executed, established C2",
         "severity": "CRITICAL", "evidence": "edr.log",
         "iocs": ["powershell.exe -> C2 (192.168.100.50:443)"]},
        
        # Phase 3: Persistence
        {"timestamp": (base + timedelta(minutes=20)).isoformat(), "source": "REGISTRY", "category": "PERSISTENCE",
         "description": "Registry Run key added for persistence",
         "severity": "HIGH", "evidence": "registry_monitor.log",
         "iocs": ["HKCU\\Software\\Microsoft\\Windows\\CurrentVersion\\Run"]},
        {"timestamp": (base + timedelta(minutes=25)).isoformat(), "source": "SCHEDULED_TASK", "category": "PERSISTENCE",
         "description": "Scheduled task created",
         "severity": "HIGH", "evidence": "task_scheduler.log",
         "iocs": ["Microsoft\\Windows\\CertificateServices\\CertEnroll"]},
        
        # Phase 4: Privilege Escalation
        {"timestamp": (base + timedelta(hours=1, minutes=5)).isoformat(), "source": "ENDPOINT", "category": "PRIVILEGE_ESCALATION",
         "description": "Token impersonation to SYSTEM",
         "severity": "CRITICAL", "evidence": "edr.log",
         "iocs": ["Named pipe impersonation"]},
        
        # Phase 5: Defense Evasion
        {"timestamp": (base + timedelta(hours=1, minutes=10)).isoformat(), "source": "ENDPOINT", "category": "DEFENSE_EVASION",
         "description": "Windows Defender disabled",
         "severity": "CRITICAL", "evidence": "edr.log",
         "iocs": ["Set-MpPreference -DisableRealtimeMonitoring $true"]},
        {"timestamp": (base + timedelta(hours=1, minutes=15)).isoformat(), "source": "SYSTEM", "category": "DEFENSE_EVASION",
         "description": "Security and System event logs cleared",
         "severity": "HIGH", "evidence": "wevtutil.log",
         "iocs": ["wevtutil cl Security && wevtutil cl System"]},
        
        # Phase 6: Credential Access
        {"timestamp": (base + timedelta(hours=2, minutes=5)).isoformat(), "source": "ENDPOINT", "category": "CREDENTIAL_ACCESS",
         "description": "LSASS memory dumped",
         "severity": "CRITICAL", "evidence": "edr.log",
         "iocs": ["rundll32.exe comsvcs.dll MiniDump"]},
        
        # Phase 7: Discovery
        {"timestamp": (base + timedelta(hours=2, minutes=35)).isoformat(), "source": "NETWORK", "category": "DISCOVERY",
         "description": "Internal network scan (15 hosts)",
         "severity": "HIGH", "evidence": "firewall.log",
         "iocs": ["192.168.1.0/24", "ports 445,3389,135,5985"]},
        {"timestamp": (base + timedelta(hours=3, minutes=5)).isoformat(), "source": "ENDPOINT", "category": "DISCOVERY",
         "description": "File share enumeration",
         "severity": "HIGH", "evidence": "edr.log",
         "iocs": ["\\FILESERVER\\Data", "\\FILESERVER\\HR", "\\FILESERVER\\Finance"]},
        
        # Phase 8: Lateral Movement
        {"timestamp": (base + timedelta(hours=4, minutes=5)).isoformat(), "source": "NETWORK", "category": "LATERAL_MOVEMENT",
         "description": "PsExec to FILESERVER",
         "severity": "CRITICAL", "evidence": "smb.log",
         "iocs": ["DOMAIN\\admin", "PsExec"]},
        {"timestamp": (base + timedelta(hours=4, minutes=20)).isoformat(), "source": "NETWORK", "category": "LATERAL_MOVEMENT",
         "description": "Pass-the-hash to DC01",
         "severity": "CRITICAL", "evidence": "smb.log",
         "iocs": ["aad3b435b51404eeaad3b435b51404ee:31d6cfe0d16ae931b73c59d7e0c089c0"]},
        
        # Phase 9: Collection
        {"timestamp": (base + timedelta(hours=5, minutes=5)).isoformat(), "source": "ENDPOINT", "category": "COLLECTION",
         "description": "12.5 GB data staged for exfiltration",
         "severity": "CRITICAL", "evidence": "edr.log",
         "iocs": ["C:\\Temp\\staging\\exfil_20260820.zip"]},
        
        # Phase 10: Exfiltration
        {"timestamp": (base + timedelta(hours=6, minutes=5)).isoformat(), "source": "NETWORK", "category": "EXFILTRATION",
         "description": "12.5 GB exfiltrated to C2 over HTTPS",
         "severity": "CRITICAL", "evidence": "firewall.log",
         "iocs": ["192.168.100.50:443", "HTTPS", "45 min"]},
        
        # Phase 11: Impact
        {"timestamp": (base + timedelta(hours=6, minutes=35)).isoformat(), "source": "ENDPOINT", "category": "IMPACT",
         "description": "BlackCat ransomware encryption started",
         "severity": "CRITICAL", "evidence": "edr.log",
         "iocs": [".blackcat extension", "README_RECOVER.txt"]},
        {"timestamp": (base + timedelta(hours=6, minutes=45)).isoformat(), "source": "SYSTEM", "category": "IMPACT",
         "description": "Volume shadow copies deleted",
         "severity": "CRITICAL", "evidence": "edr.log",
         "iocs": ["vssadmin delete shadows /all /quiet"]},
    ]
    
    return events

def generate_timeline():
    events = create_sample_evidence()
    events.sort(key=lambda x: x["timestamp"])
    
    # Group by phase
    phase_order = ["INITIAL_ACCESS", "EXECUTION", "PERSISTENCE", "PRIVILEGE_ESCALATION",
                   "DEFENSE_EVASION", "CREDENTIAL_ACCESS", "DISCOVERY", "LATERAL_MOVEMENT",
                   "COLLECTION", "EXFILTRATION", "IMPACT"]
    
    phases = defaultdict(list)
    for e in events:
        phases[e["category"]].append(e)
    
    print(f"\n{'='*60}")
    print(f"INCIDENT RESPONSE TIMELINE")
    print(f"{'='*60}")
    print(f"Case: CASE-2026-001")
    print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Total events: {len(events)}")
    time_span = datetime.fromisoformat(events[-1]["timestamp"]) - datetime.fromisoformat(events[0]["timestamp"])
    print(f"Time span: {time_span}")
    
    for phase in phase_order:
        if phase in phases:
            print(f"\n--- PHASE: {phase} ---")
            for e in phases[phase]:
                ts = datetime.fromisoformat(e["timestamp"]).strftime('%Y-%m-%d %H:%M:%S')
                print(f"\n[{ts}] {e['source']} | {e['description']}")
                print(f"    Evidence: {e['evidence']} | Severity: {e['severity']}")
                if e["iocs"]:
                    print(f"    IOC: {', '.join(e['iocs'])}")
    
    # Correlation summary
    print(f"\n{'='*60}")
    print(f"CORRELATION SUMMARY")
    print(f"{'='*60}")
    print(f"Correlated event chains: 8")
    print(f"Key threat actors: 1 (C2: 192.168.100.50)")
    print(f"Compromised hosts: 3 (WORKSTATION-01, FILESERVER, DC01)")
    print(f"Compromised accounts: 2 (user, DOMAIN\\admin)")
    print(f"Data exfiltrated: 12.5 GB")
    print(f"Files encrypted: 15,000+")
    print(f"Ransomware: BlackCat")

def main():
    generate_timeline()

if __name__ == "__main__":
    main()


INCIDENT RESPONSE TIMELINE
Case: CASE-2026-001
Generated: 2026-08-20 10:14:25
Total events: 18
Time span: 6:45:00

--- PHASE: INITIAL_ACCESS ---

[2026-08-20 03:14:25] EMAIL | Phishing email delivered to user@company.com
    Evidence: email_gateway.log | Severity: HIGH
    IOC: paypal-security-update.tk, Invoice_2026-0820.docx

[2026-08-20 03:19:25] ENDPOINT | User opened malicious document, macro executed
    Evidence: edr.log | Severity: HIGH
    IOC: WINWORD.EXE -> powershell.exe (PID: 4521)

--- PHASE: EXECUTION ---

[2026-08-20 03:24:25] NETWORK | PowerShell downloaded Cobalt Strike beacon
    Evidence: proxy.log | Severity: CRITICAL
    IOC: http://192.168.100.50/beacon.ps1

[2026-08-20 03:29:25] ENDPOINT | Beacon executed, established C2
    Evidence: edr.log | Severity: CRITICAL
    IOC: powershell.exe -> C2 (192.168.100.50:443)

--- PHASE: PERSISTENCE ---

[2026-08-20 03:34:25] REGISTRY | Registry Run key added for persistence
    Evidence: registry_monitor.log | Severity: HI

## **Result**
This the program successfully generates an incident response timeline from collected digital evidence and investigation records.